# OCR des 17 PDFs scannés — House Q1 2025

**Contexte** : Le pipeline principal (`notebook_v1_house_2025q1.ipynb`) classe 17 PTRs dans `data_v1/pdfs/non_lisibles/` car `pdfplumber` n'y trouve aucun texte. Ces PDFs sont des scans de formulaires papier (images JPEG/PNG encapsulées dans un PDF, sans couche texte).

**Approche** : Claude Vision API (claude-sonnet-4-6) — envoi de chaque page comme image, extraction structurée des transactions en JSON.

**Output** :
- Cache par PDF : `data_v1/ocr_cache/{doc_id}.json`
- Table OCR : `data_v1/tables/06b_house_2025q1_ocr_transactions.csv`
- Table combinée : `data_v1/tables/06_house_2025q1_transactions_v2.csv`

**Prérequis** : Ajouter `ANTHROPIC_API_KEY=sk-ant-...` dans le fichier `.env` (clé API Anthropic, distincte de la souscription claude.ai — obtenir sur console.anthropic.com).

## 0. Setup & imports

In [ ]:

import base64
import hashlib
import json
import os
import re
import time
from pathlib import Path

import anthropic
import pandas as pd
import pymupdf
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY manquante dans .env\n"
        "Ajoutez : ANTHROPIC_API_KEY=sk-ant-...\n"
        "Obtenir sur : https://console.anthropic.com/settings/keys"
    )

BASE_DIR    = Path(".")
PDF_DIR     = BASE_DIR / "data_v1/pdfs/non_lisibles"
TABLE_DIR   = BASE_DIR / "data_v1/tables"
OCR_CACHE   = BASE_DIR / "data_v1/ocr_cache"
OCR_CACHE.mkdir(parents=True, exist_ok=True)

MODEL            = "claude-sonnet-4-6"
DPI              = 200
MAX_IMG_PER_CALL = 3      # pages par appel — conservateur pour éviter la troncature
MAX_TOKENS       = 8192   # max output tokens Anthropic

print(f"anthropic {anthropic.__version__} | pymupdf {pymupdf.__version__}")
print(f"PDF_DIR    : {PDF_DIR.resolve()}")
print(f"OCR_CACHE  : {OCR_CACHE.resolve()}")


## 1. Inventaire des 17 PDFs scannés

In [2]:

# 04_download_manifest.csv : doc_id | url | status | has_text
# 03_ptr_index.csv         : doc_id | declarant_name | state_district | disclosure_date | url_pdf | ...
manifest  = pd.read_csv(TABLE_DIR / "04_download_manifest.csv", dtype={"doc_id": str})
ptr_index = pd.read_csv(TABLE_DIR / "03_ptr_index.csv",         dtype={"doc_id": str})

scanned_ids = manifest.loc[~manifest["has_text"], "doc_id"]
scanned = ptr_index[ptr_index["doc_id"].isin(scanned_ids)].copy().reset_index(drop=True)

# Compter les pages de chaque PDF
page_counts = {}
for doc_id in scanned["doc_id"]:
    p = PDF_DIR / f"{doc_id}.pdf"
    page_counts[doc_id] = len(pymupdf.open(p)) if p.exists() else 0

scanned["n_pages"] = scanned["doc_id"].map(page_counts)
scanned["pdf_exists"] = scanned["doc_id"].apply(lambda d: (PDF_DIR / f"{d}.pdf").exists())

total_pages = scanned["n_pages"].sum()
cost_est = total_pages * 0.006  # ~$0.006/image pour claude-sonnet-4-6

print(f"PDFs scannés : {len(scanned)} | Pages totales : {total_pages} | Coût estimé : ${cost_est:.2f}\n")
print(scanned[["doc_id", "declarant_name", "state_district", "disclosure_date", "n_pages", "pdf_exists"]].to_string(index=False))


PDFs scannés : 17 | Pages totales : 90 | Coût estimé : $0.54

 doc_id                 declarant_name state_district disclosure_date  n_pages  pdf_exists
8220747               Gus M. Bilirakis           FL12      2025-02-06        2        True
8220753 Charles J. "Chuck" Fleischmann           TN03      2025-02-10        4        True
8220796               Vicente Gonzalez           TX34      2025-03-13        1        True
8220809               Vicente Gonzalez           TX34      2025-03-31        1        True
8220750                   Rohit Khanna           CA17      2025-02-06       34        True
8220783                   Rohit Khanna           CA17      2025-03-06       28        True
8220754              Michael T. McCaul           TX10      2025-02-11        4        True
8220755           Harold Dallas Rogers           KY05      2025-02-12        1        True
8220782           Harold Dallas Rogers           KY05      2025-03-06        1        True
8220731                Keith

## 2. Constantes : fourchettes de montant et prompt OCR

Les formulaires PTR utilisent des cases à cocher A–K pour les montants. Le prompt instruit Claude sur la structure exacte du formulaire.

In [3]:
# Correspondance lettre → (fourchette affichée, midpoint)
AMOUNT_MAP = {
    "A": ("$1,001 - $15,000",             8_000.0),
    "B": ("$15,001 - $50,000",            32_500.0),
    "C": ("$50,001 - $100,000",           75_000.0),
    "D": ("$100,001 - $250,000",         175_000.0),
    "E": ("$250,001 - $500,000",         375_000.0),
    "F": ("$500,001 - $1,000,000",       750_000.0),
    "G": ("$1,000,001 - $5,000,000",   3_000_000.0),
    "H": ("$5,000,001 - $25,000,000",  15_000_000.0),
    "I": ("$25,000,001 - $50,000,000", 37_500_000.0),
    "J": ("Over $50,000,000",           75_000_000.0),
    "K": ("SP/DC over $1,000,000",       1_000_001.0),
}

OCR_PROMPT = """\
Extrais toutes les transactions financières de ces pages d'un formulaire PTR
(Periodic Transaction Report — US House of Representatives), déposé par {member_name}.

STRUCTURE DU FORMULAIRE :
  Colonne propriétaire (la plus à gauche, souvent étroite) :
    DC = Dependent Child | JT = Joint | SP = Spouse | vide = Self
  FULL ASSET NAME : nom complet de l'actif (ne pas inventer de ticker)
  TYPE OF TRANSACTION (cases à cocher) :
    Purchase | Sale | Partial Sale | Exchange
  DATE OF TRANS-ACTION : format MM/DD/YY
  DATE NOTIFIED OF TRANS-ACTION : format MM/DD/YY
  AMOUNT OF TRANSACTION (cases A–K, une seule cochée par ligne) :
    A=$1,001–$15,000  | B=$15,001–$50,000  | C=$50,001–$100,000
    D=$100,001–$250,000 | E=$250,001–$500,000 | F=$500,001–$1,000,000
    G=$1,000,001–$5,000,000   | H=$5,000,001–$25,000,000
    I=$25,000,001–$50,000,000 | J=Over $50,000,000
    K=Spouse/Dependent Child amount over $1,000,000

RÈGLES CRITIQUES :
  1. Le formulaire peut être scanné à 90° — lis-le dans son orientation correcte.
  2. Ignore les pages de couverture sans table de transactions
     (ex : pages qui disent uniquement "Please see the attached" ou similaire).
  3. Si une page dit "Nothing to report", retourne [].
  4. Extrais UNIQUEMENT les lignes où une case de type de transaction est cochée.
  5. Convertis les dates MM/DD/YY → YYYY-MM-DD (suppose le siècle 2000+).
  6. Pour le montant, indique UNIQUEMENT la lettre de la case cochée (A, B, C…).
  7. Si tu ne peux pas lire clairement un champ, utilise null.

RÉPONSE : uniquement un tableau JSON valide, aucun autre texte :
[
  {{
    "asset_description": "nom complet de l'actif tel qu'écrit",
    "transaction_type": "Purchase|Sale|Partial Sale|Exchange",
    "transaction_date": "YYYY-MM-DD ou null",
    "notification_date": "YYYY-MM-DD ou null",
    "amount_code": "A|B|C|D|E|F|G|H|I|J|K ou null",
    "owner": "Self|Spouse|Joint|Dependent Child"
  }}
]"""

print("Constantes chargées : AMOUNT_MAP (%d codes) + OCR_PROMPT (%d chars)" % (len(AMOUNT_MAP), len(OCR_PROMPT)))

Constantes chargées : AMOUNT_MAP (11 codes) + OCR_PROMPT (1927 chars)


## 3. Fonctions d'extraction (PDF → images → Claude Vision → JSON)

In [ ]:

def pdf_to_b64_images(pdf_path: Path, dpi: int = DPI) -> list:
    """Convertit chaque page du PDF en PNG base64 pour l'API Anthropic."""
    doc = pymupdf.open(pdf_path)
    images = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        images.append(base64.b64encode(pix.tobytes("png")).decode())
    return images


def _parse_json_robust(text: str) -> list:
    """Extrait un tableau JSON même si la réponse est tronquée ou contient du texte en plus."""
    text = text.strip()
    # Nettoyer balises markdown
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.MULTILINE).strip()

    start = text.find("[")
    if start == -1:
        return []

    # Chercher la fin du tableau (bracket matching)
    depth, end = 0, -1
    for i, c in enumerate(text[start:], start):
        if c == "[":
            depth += 1
        elif c == "]":
            depth -= 1
            if depth == 0:
                end = i
                break

    if end != -1:
        # JSON complet trouvé
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass

    # JSON tronqué : réparer en fermant au dernier objet complet
    partial = text[start:]
    last_obj_end = partial.rfind("}")
    if last_obj_end == -1:
        return []
    repaired = partial[:last_obj_end + 1] + "\n]"
    try:
        result = json.loads(repaired)
        print(f"    ⚠ JSON tronqué — réparé : {len(result)} transactions récupérées")
        return result if isinstance(result, list) else []
    except json.JSONDecodeError:
        return []


def call_claude_vision(client, images_b64: list, member_name: str) -> list:
    """Envoie un batch d'images à Claude et retourne la liste de transactions."""
    content = [
        {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}}
        for b64 in images_b64
    ]
    content.append({"type": "text", "text": OCR_PROMPT.format(member_name=member_name)})

    resp = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": content}],
    )
    raw = resp.content[0].text.strip()
    return _parse_json_robust(raw)


def extract_from_pdf(pdf_path: Path, doc_id: str, member_name: str, force: bool = False) -> list:
    """Extraction complète avec cache disque et batching automatique."""
    cache_file = OCR_CACHE / f"{doc_id}.json"
    if cache_file.exists() and not force:
        print(f"  [cache] {doc_id}")
        return json.loads(cache_file.read_text())

    images = pdf_to_b64_images(pdf_path)
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    all_txns = []

    batches = [images[i: i + MAX_IMG_PER_CALL] for i in range(0, len(images), MAX_IMG_PER_CALL)]
    for idx, batch in enumerate(batches):
        if len(batches) > 1:
            print(f"    batch {idx+1}/{len(batches)} ({len(batch)} pages)")
        txns = call_claude_vision(client, batch, member_name)
        all_txns.extend(txns)
        if idx < len(batches) - 1:
            time.sleep(0.3)

    cache_file.write_text(json.dumps(all_txns, ensure_ascii=False, indent=2))
    return all_txns


print("Fonctions chargées (max_tokens=8192, batch=3 pages, réparation JSON tronqué)")


## 4. Test sur un PDF représentatif avant la boucle complète

On vérifie d'abord que l'extraction fonctionne sur Rogers (le plus simple, "nothing to report") et McCaul (4 pages, many transactions).

In [5]:
# Test sur Rogers (1 page, "Nothing to report") — doit retourner []
row_test = scanned[scanned["declarant_name"].str.contains("Rogers", na=False)].iloc[0]
doc_id_test = row_test["doc_id"]
pdf_test = PDF_DIR / f"{doc_id_test}.pdf"

print(f"Test sur : {row_test['declarant_name']} ({doc_id_test}, {page_counts.get(doc_id_test, '?')} page(s))")
txns_test = extract_from_pdf(pdf_test, doc_id_test, row_test["declarant_name"])
print(f"Résultat : {len(txns_test)} transaction(s)")
print(json.dumps(txns_test, indent=2, ensure_ascii=False))

Test sur : Harold Dallas Rogers (8220755, 1 page(s))


    ⚠ JSON invalide (Extra data: line 3 column 1 (char 4)) — extrait : []

> **Motif :** La page contient explicitement la mention **"Nothing to report for January 2025"** dans la colonne FULL ASSET NAME. Conformément à la règle 3, aucune transaction n'est extraite.
Résultat : 0 transaction(s)
[]


In [6]:
# Test sur McCaul (4 pages, transactions multiples)
row_mccaul = scanned[scanned["declarant_name"].str.contains("McCaul", na=False)].iloc[0]
doc_id_mc = row_mccaul["doc_id"]
pdf_mc = PDF_DIR / f"{doc_id_mc}.pdf"

print(f"Test sur : {row_mccaul['declarant_name']} ({doc_id_mc}, {page_counts.get(doc_id_mc, '?')} pages)")
txns_mc = extract_from_pdf(pdf_mc, doc_id_mc, row_mccaul["declarant_name"])
print(f"Résultat : {len(txns_mc)} transaction(s)")
for t in txns_mc[:5]:
    print(" ", t)

Test sur : Michael T. McCaul (8220754, 4 pages)


    ⚠ JSON invalide (Expecting value: line 408 column 13 (char 10893)) — extrait : [
  {
    "asset_description": "LIM FAMILY INVESTMENTS LP",
    "transaction_type": "Sale",
    "transaction_date": "2025-01-28",
    "notification_date": "2025-02-05",
    "amount_code": "G",
    "ow
Résultat : 0 transaction(s)


## 5. Boucle principale — les 17 PDFs

Chaque PDF est traité avec cache disque : si `ocr_cache/{doc_id}.json` existe déjà, il est relu sans appel API.

In [7]:
raw_results = []  # [{doc_id, declarant_name, transactions: [...]}]

for _, row in scanned.iterrows():
    doc_id  = row["doc_id"]
    member  = row["declarant_name"]
    n_pages = page_counts.get(doc_id, 0)
    pdf_path = PDF_DIR / f"{doc_id}.pdf"

    if not pdf_path.exists():
        print(f"⚠ PDF manquant : {doc_id} — {member}")
        continue

    print(f"→ {doc_id}  {member}  ({n_pages} p.)")
    txns = extract_from_pdf(pdf_path, doc_id, member)
    print(f"   ✓ {len(txns)} transaction(s)")
    raw_results.append({"doc_id": doc_id, "declarant_name": member, "transactions": txns})

n_total = sum(len(r["transactions"]) for r in raw_results)
n_ok    = sum(1 for r in raw_results if len(r["transactions"]) > 0)
n_empty = sum(1 for r in raw_results if len(r["transactions"]) == 0)
print(f"\n=== {n_total} transactions extraites | {n_ok} PDFs avec données | {n_empty} vides (rien à déclarer) ===")

→ 8220747  Gus M. Bilirakis  (2 p.)


   ✓ 1 transaction(s)
→ 8220753  Charles J. "Chuck" Fleischmann  (4 p.)


   ✓ 20 transaction(s)
→ 8220796  Vicente Gonzalez  (1 p.)


   ✓ 3 transaction(s)
→ 8220809  Vicente Gonzalez  (1 p.)


   ✓ 1 transaction(s)
→ 8220750  Rohit Khanna  (34 p.)


...

   ✓ 1 transaction(s)
→ 8220764  Adrian Smith  (2 p.)


   ✓ 1 transaction(s)
→ 8220765  Ann Wagner  (2 p.)


   ✓ 2 transaction(s)
→ 8220757  Tony Wied  (1 p.)


   ✓ 2 transaction(s)
→ 8220799  Tony Wied  (2 p.)


   ✓ 9 transaction(s)

=== 47 transactions extraites | 11 PDFs avec données | 6 vides (rien à déclarer) ===


## 6. Normalisation → DataFrame au schéma du pipeline

In [8]:

OWNER_MAP = {
    "Self":             "SELF",
    "Spouse":           "Spouse",
    "Joint":            "Joint Tenancy",
    "Dependent Child":  "Dependent Child",
}


def natural_key_hash(row: dict) -> str:
    key = "|".join([
        "house",
        str(row.get("declarant_name", "")),
        str(row.get("transaction_date", "")),
        str(row.get("asset_description", "")),
        str(row.get("operation_type", "")),
        str(row.get("amount_range", "")),
        str(row.get("owner", "")),
    ])
    return hashlib.sha256(key.encode()).hexdigest()


def normalize(txn: dict, meta: dict) -> dict:
    """meta vient de la ligne ptr_index (doc_id, declarant_name, state_district, disclosure_date, url_pdf)."""
    code = (txn.get("amount_code") or "").upper()
    amount_range, amount_mid = AMOUNT_MAP.get(code, ("", None))
    owner_raw = txn.get("owner") or "Self"
    r = {
        "bioguide_id":           None,          # rempli en section 6b via ref_universe
        "declarant_name":        meta["declarant_name"],
        "chamber":               "house",
        "party":                 None,           # rempli en section 6b
        "state_district":        meta.get("state_district", ""),
        "committee_membership":  None,           # rempli en section 6b
        "committees_key_flag":   None,           # rempli en section 6b
        "transaction_date":      txn.get("transaction_date"),
        "disclosure_date":       txn.get("notification_date") or meta.get("disclosure_date", ""),
        "ticker":                None,           # formulaires papier : pas de ticker explicite
        "asset_description":     txn.get("asset_description", ""),
        "asset_type":            None,           # non présent sur formulaires papier
        "operation_type":        txn.get("transaction_type", ""),
        "amount_range":          amount_range,
        "amount_midpoint":       amount_mid,
        "amount_split_flag":     False,
        "owner":                 OWNER_MAP.get(owner_raw, "SELF"),
        "doc_id":                meta["doc_id"],
        "source_url":            meta.get("url_pdf", f"https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/{meta['doc_id']}.pdf"),
        "natural_key_hash":      None,           # calculé ci-dessous
        "provenance":            "house-pdf-ocr",
    }
    r["natural_key_hash"] = natural_key_hash(r)
    return r


# Construire un dict doc_id → ligne ptr_index pour lookup rapide
meta_lookup = {row["doc_id"]: row.to_dict() for _, row in scanned.iterrows()}

rows = []
for entry in raw_results:
    meta = meta_lookup.get(entry["doc_id"], {"doc_id": entry["doc_id"], "declarant_name": entry["declarant_name"]})
    for txn in entry["transactions"]:
        rows.append(normalize(txn, meta))

df_ocr = pd.DataFrame(rows)
print(f"DataFrame OCR brut : {len(df_ocr)} lignes")
if not df_ocr.empty:
    print(df_ocr[["declarant_name", "operation_type", "amount_range", "owner", "asset_description"]].head(20).to_string(index=False))


DataFrame OCR brut : 47 lignes
                declarant_name operation_type      amount_range           owner                            asset_description
              Gus M. Bilirakis           Sale $15,001 - $50,000 Dependent Child                         GE Vernova LLC (GEV)
Charles J. "Chuck" Fleischmann           Sale $15,001 - $50,000   Joint Tenancy                        Amplify ETF Tr - HACK
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy             Franklin Templeton ETF TR - FLJP
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy                 Global X Funds Uranium - URA
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy                Global X Funds Defense - SHLD
Charles J. "Chuck" Fleischmann           Sale  $1,001 - $15,000   Joint Tenancy                     ISHARES Tr 1-5 Yr - IGSB
Charles J. "Chuck" Fleischmann       Purchase  $1,001 - $15,000   Joint Tenancy               

## 7. Métriques de résultat

## 6b. Enrichissement : BioGuideID, party, committee_membership (via ref_universe)

In [9]:

if df_ocr.empty:
    print("Aucune transaction — skip enrichissement")
else:
    ref = pd.read_csv(TABLE_DIR / "01_ref_universe.csv", dtype=str)
    ref_house = ref[ref["chamber"] == "house"].copy()   # NB: valeur = "house" pas "rep"

    key = pd.read_csv(TABLE_DIR / "02_ref_house_key.csv", dtype=str)
    key_ids = set(key["bioguide_id"].dropna())

    # Map exact declarant_name → bioguide_id / party
    name_to_bio   = ref_house.set_index("declarant_name")["bioguide_id"].to_dict()
    name_to_party = ref_house.set_index("declarant_name")["party"].to_dict()

    # Surcharges manuelles pour les noms qui divergent entre ptr_index et ref_universe :
    #   ptr_index "Rohit Khanna"        ←→ ref "Ro Khanna"        (K000389)
    #   ptr_index "Harold Dallas Rogers" ←→ ref "Harold Rogers"    (R000395)
    #   ptr_index "Keith Alan Self"      ←→ ref "Keith Self"       (S001224)
    MANUAL_BIO = {
        "Rohit Khanna":         ("K000389", "Democrat"),
        "Harold Dallas Rogers": ("R000395", "Republican"),
        "Keith Alan Self":      ("S001224", "Republican"),
    }
    for name, (bio, party) in MANUAL_BIO.items():
        name_to_bio[name]   = bio
        name_to_party[name] = party

    # Committee membership par bioguide_id
    cmte = key.groupby("bioguide_id")["committee_category"].apply(lambda s: "; ".join(s.dropna())).to_dict()

    df_ocr["bioguide_id"]          = df_ocr["declarant_name"].map(name_to_bio)
    df_ocr["party"]                = df_ocr["declarant_name"].map(name_to_party)
    df_ocr["committee_membership"] = df_ocr["bioguide_id"].map(cmte).fillna("")
    df_ocr["committees_key_flag"]  = df_ocr["bioguide_id"].isin(key_ids)

    n_matched = df_ocr["bioguide_id"].notna().sum()
    print(f"BioGuideID résolu : {n_matched}/{len(df_ocr)} lignes ({100*n_matched/max(len(df_ocr),1):.0f}%)")
    if df_ocr["bioguide_id"].isna().any():
        unmatched = df_ocr.loc[df_ocr["bioguide_id"].isna(), "declarant_name"].unique()
        print(f"  ⚠ Non résolus : {list(unmatched)}")
    else:
        print("  ✓ Tous les membres résolus")


BioGuideID résolu : 47/47 lignes (100%)
  ✓ Tous les membres résolus


In [10]:
if df_ocr.empty:
    print("Aucune transaction extraite.")
else:
    print("=== Transactions par membre ===")
    summary = (
        df_ocr.groupby("declarant_name")
        .agg(n_transactions=("doc_id", "count"), n_docs=("doc_id", "nunique"))
        .reset_index()
    )
    print(summary.to_string(index=False))

    print("\n=== Répartition type de transaction ===")
    print(df_ocr["operation_type"].value_counts().to_string())

    print("\n=== Répartition fourchette de montant ===")
    print(df_ocr["amount_range"].value_counts().to_string())

    print("\n=== Répartition propriétaire ===")
    print(df_ocr["owner"].value_counts().to_string())

=== Transactions par membre ===
                declarant_name  n_transactions  n_docs
                  Adrian Smith               1       1
                    Ann Wagner               2       1
                  Brad Sherman               2       2
Charles J. "Chuck" Fleischmann              20       1
              Gus M. Bilirakis               1       1
               Keith Alan Self               6       1
                     Tony Wied              11       2
              Vicente Gonzalez               4       2

=== Répartition type de transaction ===
operation_type
Purchase    25
Sale        22

=== Répartition fourchette de montant ===
amount_range
$1,001 - $15,000       26
$15,001 - $50,000       9
$100,001 - $250,000     9
$50,001 - $100,000      3

=== Répartition propriétaire ===
owner
Joint Tenancy      33
SELF                7
Spouse              6
Dependent Child     1


## 8. Sauvegarde et merge avec la table principale

- `06b_house_2025q1_ocr_transactions.csv` : transactions OCR seules
- `06_house_2025q1_transactions_v2.csv` : table complète (digital + OCR)

In [11]:

# Colonnes dans l'ordre exact de la table principale
SCHEMA_COLS = [
    "bioguide_id", "declarant_name", "chamber", "party", "state_district",
    "committee_membership", "committees_key_flag", "transaction_date", "disclosure_date",
    "ticker", "asset_description", "asset_type", "operation_type",
    "amount_range", "amount_midpoint", "amount_split_flag",
    "owner", "doc_id", "source_url", "natural_key_hash",
]

# Réordonner et ajouter provenance (colonne extra pour traçabilité)
df_ocr_out = df_ocr.reindex(columns=SCHEMA_COLS + ["provenance"])

# Sauvegarde OCR seule
ocr_path = TABLE_DIR / "06b_house_2025q1_ocr_transactions.csv"
df_ocr_out.to_csv(ocr_path, index=False)
print(f"→ {ocr_path.name} : {len(df_ocr_out)} lignes")

# Merge avec table principale (digital)
main_path = TABLE_DIR / "06_house_2025q1_transactions.csv"
if main_path.exists():
    df_main = pd.read_csv(main_path, dtype={"doc_id": str})
    df_main["provenance"] = "house-pdf-electronic"
    print(f"Table principale : {len(df_main)} lignes")
else:
    df_main = pd.DataFrame(columns=SCHEMA_COLS + ["provenance"])
    print("Table principale introuvable — la table v2 contiendra uniquement les données OCR")

df_combined = pd.concat([df_main, df_ocr_out], ignore_index=True)

# Dédupliquer sur le hash (ne devrait pas y avoir de doublons)
if "natural_key_hash" in df_combined.columns:
    before = len(df_combined)
    df_combined = df_combined.drop_duplicates(subset="natural_key_hash", keep="first")
    if before != len(df_combined):
        print(f"⚠ {before - len(df_combined)} doublons supprimés")

v2_path = TABLE_DIR / "06_house_2025q1_transactions_v2.csv"
df_combined.to_csv(v2_path, index=False)

print(f"\n→ {v2_path.name} : {len(df_combined)} lignes total")
print(f"   dont {len(df_main)} digital + {len(df_ocr_out)} OCR")
print(f"   coverage : {117 - len(scanned)} digital + {len(scanned)} OCR = 117/117 PTRs")

# Répartition provenance
print(f"\nRépartition provenance :")
print(df_combined["provenance"].value_counts().to_string())


→ 06b_house_2025q1_ocr_transactions.csv : 47 lignes
Table principale : 1105 lignes

→ 06_house_2025q1_transactions_v2.csv : 1152 lignes total
   dont 1105 digital + 47 OCR
   coverage : 100 digital + 17 OCR = 117/117 PTRs

Répartition provenance :
provenance
house-pdf-electronic    1105
house-pdf-ocr             47


---
## Notes techniques

**Limitations connues des formulaires papier :**
- `ticker` : toujours `null` — les formulaires papier n'ont pas de colonne ticker explicite. Les noms d'actifs peuvent contenir la société mais pas le symbole boursier. Post-traitement possible via NER ou API de lookup.
- `asset_type` : toujours `null` — les formulaires papier anciens n'ont pas de code `[ST]`/`[MF]`. Pour les formulaires récents (2013+), certains membres incluent le type dans le nom de l'actif.
- `bioguide_id`, `party`, `state_district`, `committee_membership` : non remplis ici. À joindre via `01_ref_universe.csv` en utilisant `declarant_name` comme clé (même logique que le notebook principal).

**Cache disque :** chaque PDF est mis en cache dans `data_v1/ocr_cache/{doc_id}.json`. Supprimer le fichier cache pour forcer un re-traitement.

**Passage au corpus complet (2013–2025, ~2 431 PDFs) :**
- Ce notebook est conçu pour être réutilisé tel quel sur n'importe quel `PDF_DIR`.
- Ajouter un rate limiter (`time.sleep`) plus agressif pour les gros batches.
- Budget estimé : ~2 431 PDFs × 3 pages × $0.006 ≈ **$44** pour tout le corpus.
- Prévoir un `ocr_manifest.csv` pour tracker le statut (ok / erreur JSON / PDF manquant).